In [ ]:
def forwards(
        img: pnex.In("f32", (None, meta.img_cs, meta.img_ys, meta.img_xs)),
        kernel: pnex.In("f32", (meta.krn_os, meta.krn_cs, meta.krn_ys, meta.krn_xs)),
        out_img: pnex.Out("f32", ("img", meta.out_cs, meta.out_ys, meta.out_xs)),
        out_prov: pnex.Out(
            prov_t.torch_type,
            (
                    "img.shape[0]",
                    meta.out_cs,
                    meta.out_ys,
                    meta.out_xs,
                    3 if meta.krn_cs > 1 else 2,
            ),
        ),
):
    rem, o_x = divmod(cuda.grid(1), meta.out_xs)
    rem, o_y = divmod(rem, meta.out_ys)
    b, o_c = divmod(rem, meta.out_cs)
    if b >= img.shape[0]:
        return

    i_top_y = o_y * meta.str_y - meta.pad_y_beg
    i_left_x = o_x * meta.str_x - meta.pad_x_beg

    prov_x = prov_y = prov_group_idx = prov_t.maxval
    selected_val = numba.float32(semifield.zero)

    group_number = o_c // meta.grp_o
    # If we're not broadcasting, then we have a separate kernel
    # for every output channel. If we are broadcasting, we instead loop
    # around the kernels every k_os (which == krn_group_size)
    k_o = o_c if not meta.group_broadcasting else o_c % meta.grp_o

    # For a pooling, we have only one input channel, so group_idx is always 0
    for group_idx in range(meta.krn_cs):
        for y_step, i_y in enumerate(
                range(i_top_y, i_top_y + meta.krn_ys * meta.dil_y, meta.dil_y)
        ):
            for x_step, i_x in enumerate(
                    range(
                        i_left_x,
                        i_left_x + meta.krn_xs * meta.dil_x,
                        meta.dil_x,
                    )
            ):
                if i_x < 0 or i_x >= meta.img_xs or i_y < 0 or i_y >= meta.img_ys:
                    continue

                # Need to explicitly use seperate variable, due to compiler error
                if meta.mirror_kernel:
                    k_x = meta.krn_xs - 1 - x_step
                    k_y = meta.krn_ys - 1 - y_step
                else:
                    k_x = x_step
                    k_y = y_step

                i_c = group_number * meta.krn_cs + group_idx
                img_val = img[b, i_c, i_y, i_x]
                kernel_val = kernel[k_o, group_idx, k_y, k_x]

                val = semifield.times(img_val, kernel_val)
                if semifield.add_select(selected_val, val):
                    selected_val = val
                    prov_y, prov_x = k_y, k_x
                    if meta.krn_cs > 1:
                        prov_group_idx = group_idx

    out_img[b, o_c, o_y, o_x] = selected_val

    out_prov[b, o_c, o_y, o_x, 0] = prov_y
    out_prov[b, o_c, o_y, o_x, 1] = prov_x
    if meta.krn_cs > 1:
        # out_prov is only size 3 if we require an index within the group
        out_prov[b, o_c, o_y, o_x, 2] = prov_group_idx